In [52]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px


In [ ]:
#lecture image et mask
img = cv2.imread('data/image1_base.png')
mask = cv2.imread('data/mask1.png', 0)

#invertion masque
mask = cv2.bitwise_not(mask)


#affichage
mask_3ch = cv2.merge([mask, mask, mask]) #passage niveau gris vers RGB pour affichage
combined = np.stack([cv2.cvtColor(img, cv2.COLOR_BGR2RGB), mask_3ch])
fig = px.imshow(combined, facet_col=0, binary_string=True)
fig.layout.annotations[0].text = "Image Originale"
fig.layout.annotations[1].text = "Masque"
fig.update_layout(showlegend=False)
fig.show()



In [54]:
#redimension
mask = cv2.resize(mask, (img.shape[1], img.shape[0]))

#sécu binarisation mask
_, mask_bin = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

#image avec mask
img_m = cv2.bitwise_and(img, img, mask=mask_bin)
fig = px.imshow(cv2.cvtColor(img_m, cv2.COLOR_BGR2RGB))
fig.show()

In [55]:

k_3x3 = np.array([[1, 1, 1],
              [1, 0, 1],
              [1, 1, 1]])
k_3x3 = k_3x3 / k_3x3.sum()

k_5x5 = np.array([
    [1, 2,  4, 2, 1],
    [2, 4,  8, 4, 2],
    [4, 8,  0, 8, 4],  
    [2, 4,  8, 4, 2],
    [1, 2,  4, 2, 1]
])  
k_5x5 = k_5x5 / k_5x5.sum() 


# Fonction de Priorité

In [56]:
def priorité(patch_grad_x, patch_grad_y, patch_C, normale_p, R):
    # C(p)
    surface = (2 * R + 1) ** 2
    confiance = np.sum(patch_C) / surface

    # D(p) gradient au pixel central p
    gx = patch_grad_x[R, R]
    gy = patch_grad_y[R, R]
    grad_perp = np.array([-gy, gx])
    donnee = abs(np.dot(grad_perp, normale_p)) / 255.0

    return confiance * donnee, confiance

## Init image et mask pour l'algo

In [ ]:
# Conversion Bgr rgb a cause opencv 
img_m_rgb = cv2.cvtColor(img_m, cv2.COLOR_BGR2RGB)
#remise du mask pour les calculs
mask = cv2.bitwise_not(mask)
#affichage mask
px.imshow(mask).update_layout(title="Masque inversé").show()

## Algo de Criminisi local

In [ ]:
def algo_criminisi(image, mask, largeur_patch):
    R = (largeur_patch - 1) // 2
    h_orig, w_orig = image.shape[:2]
    
    #padding pour eviter les probleme bord si mask est proche du bord exemple img saut de l'ange
    image = cv2.copyMakeBorder(image, R, R, R, R, cv2.BORDER_REFLECT)
    mask = cv2.copyMakeBorder(mask, R, R, R, R, cv2.BORDER_CONSTANT, value=0)
    
    C = np.where(mask == 0, 1.0, 0.0)
    

    # gros_noyau = np.ones((2 * largeur_patch - 1, 2 * largeur_patch - 1), np.uint8)

    gros_noyau = np.ones((largeur_patch, largeur_patch), np.uint8)
    
    
    #dimension image
    h, w = image.shape[:2] 
    marge = largeur_patch -1
    iter_count = 0
    while np.any(mask > 0):
        #compteur iterations
       
        iter_count += 1
        print(f"Pixels restants : {np.sum(mask > 0)}")
        
        #maj grad image et normales
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY).astype(np.float32)
        grad_x_img = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        grad_y_img = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        
        
        mask_normal = C.astype(np.float32)
        nx_img = cv2.Sobel(mask_normal, cv2.CV_64F, 1, 0, ksize=3)
        ny_img = cv2.Sobel(mask_normal, cv2.CV_64F, 0, 1, ksize=3)
        
        
        zone_interdite = cv2.dilate(mask, gros_noyau, iterations=1)
        
        
        
        
        frontiere, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        points_frontiere = np.vstack(frontiere)
        liste_pixels_frontiere = points_frontiere[:, 0]
       
       #verif
        print(f"Nombre de contours : {len(frontiere)}")
        for i, c in enumerate(frontiere):
            print(f"  Contour {i} : {len(c)} points, premier point : {c[0]}")
       
       
       #init
        pix_coord = None
        prio = -1
        
        #priotité
        for x, y in liste_pixels_frontiere:
                    
            ymin = y - R
            ymax = y + R + 1
            xmin = x - R
            xmax = x + R + 1
         
         
            if not (ymin < 0 or ymax > h or xmin < 0 or xmax > w):
    
                
                nx = nx_img[y, x]
                ny = ny_img[y, x]
                norme = np.hypot(nx, ny)
                normale_mask = np.array([nx, ny]) / norme if norme > 0 else np.array([0.0, 0.0])
                    

                patch_grad_x = grad_x_img[y-R : y+R+1, x-R : x+R+1]
                patch_grad_y = grad_y_img[y-R : y+R+1, x-R : x+R+1]
                patch_confiance = C[ymin:ymax, xmin:xmax]
                
                new_prio, new_C_p = priorité(patch_grad_x, patch_grad_y, patch_confiance, normale_mask, R)
                if new_prio > prio:
                    prio = new_prio
                    C_p = new_C_p
                    pix_coord = [x, y]
    
            
        if pix_coord is None:
            print("Aucun pixel valide")
            break
        
       # Extraction patch cible
        x, y = pix_coord[0], pix_coord[1]
        ymin, ymax = y - R, y + R + 1
        xmin, xmax = x - R, x + R + 1

        patch_cible        = image[ymin:ymax, xmin:xmax]
        patch_cible_lab = cv2.cvtColor(patch_cible, cv2.COLOR_RGB2Lab).astype(np.float32) #doc : We use the CIE Lab colour space because of its property of perceptual uniformity
        patch_mask_region  = mask[ymin:ymax, xmin:xmax]
        indices_patch_mask = (patch_mask_region > 0)   # pixels à remplir
        pixels_connus      = ~indices_patch_mask        # pixels utilisables

        meilleur_score = float('inf')
        meilleur_patch = None

    
        rayon = 100  # 25 pixels de chaque côté = un carré de 50x50

        pas = 2
        
        local_ymin = max(marge, y - rayon)
        local_ymax = min(h - marge, y + rayon)
        local_xmin = max(marge, x - rayon)
        local_xmax = min(w - marge, x + rayon)

        for py in range(local_ymin, local_ymax, pas):
            for px in range(local_xmin, local_xmax, pas):
                patch_mask_corr = mask[py-R:py+R+1, px-R:px+R+1]
                if np.any(patch_mask_corr > 0):
                    continue
                patch_corr = image[py-R:py+R+1, px-R:px+R+1]

                
                patch_corr_lab  = cv2.cvtColor(patch_corr,  cv2.COLOR_RGB2Lab).astype(np.float32)
                score = np.sum((patch_cible_lab[pixels_connus] - patch_corr_lab[pixels_connus]) ** 2)

                if score < meilleur_score:
                    meilleur_score = score
                    meilleur_patch = patch_corr.copy()
      
        
        

        #verif debug
        print(f"meilleur_patch is None : {meilleur_patch is None}")
        print(f"pixels_connus sum : {pixels_connus.sum()}")
        print(f"indices_patch_mask sum : {indices_patch_mask.sum()}")
        print(f"zone_interdite pixels bloqués : {np.sum(zone_interdite > 0)} / {h*w}")
        print(f"pix_coord : {pix_coord}")

        ys, xs = np.where(mask[ymin:ymax, xmin:xmax] > 0)
        print(f"pixels à remplir (ys) : {len(ys)}")
        
        ys_abs, xs_abs = ys + ymin, xs + xmin
        image[ys_abs, xs_abs]     = meilleur_patch[ys, xs]
        C[ys_abs, xs_abs] = C_p
        mask[ys_abs, xs_abs] = 0
        
        
        mask[ys_abs, xs_abs] = 0
        print(f"mask après maj : {np.sum(mask > 0)}")
        print(f"C après maj : {np.sum(C < 1.0)}")
        
    return image[R:R+h_orig, R:R+w_orig]

## Affichage résultat

In [59]:
resultat = algo_criminisi(img, mask, largeur_patch=21)
px.imshow(img).update_layout(title="Image depart").show()
px.imshow(resultat).update_layout(title="Image réparée").show()

Pixels restants : 33952
Nombre de contours : 8
  Contour 0 : 168 points, premier point : [[ 47 419]]
  Contour 1 : 124 points, premier point : [[110 417]]
  Contour 2 : 954 points, premier point : [[391 382]]
  Contour 3 : 449 points, premier point : [[170 285]]
  Contour 4 : 340 points, premier point : [[616 269]]
  Contour 5 : 487 points, premier point : [[439  45]]
  Contour 6 : 36 points, premier point : [[610  39]]
  Contour 7 : 643 points, premier point : [[536  34]]
meilleur_patch is None : False
pixels_connus sum : 267
indices_patch_mask sum : 174
zone_interdite pixels bloqués : 68192 / 372400
pix_coord : [np.int32(689), np.int32(400)]
pixels à remplir (ys) : 174
mask après maj : 33778
C après maj : 33952
Pixels restants : 33778
Nombre de contours : 8
  Contour 0 : 168 points, premier point : [[ 47 419]]
  Contour 1 : 124 points, premier point : [[110 417]]
  Contour 2 : 954 points, premier point : [[391 382]]
  Contour 3 : 449 points, premier point : [[170 285]]
  Contour 4 : 